# 1. Modelagem de um Problema Real de Otimização

## 1.1. Contextualização

Este estudo investiga a relação histórica entre os retornos da ação VALE3 e os retornos do Ibovespa, por meio de um modelo linear estimado a partir de observações históricas de mercado.

O problema será formulado como uma minimização irrestrita da soma dos quadrados dos resíduos, buscando os parâmetros que melhor representem a relação linear observada entre as duas séries.

## 1.2. Objetivos

- Carregar e verificar os dados históricos.
- Definir as variáveis de decisão e a função objetivo.
- Deduzir analiticamente o gradiente e a matriz Hessiana.
- Analisar as propriedades de convexidade da função.
- Executar um primeiro passo de busca linear e verificar a redução da função objetivo.

## 1.3. Organização da análise

Inicialmente, serão verificadas a estrutura, a consistência das datas e a integridade das cotações. Em seguida, serão calculados os retornos utilizados na modelagem.

A formulação matemática, as deduções analíticas e os resultados serão desenvolvidos progressivamente, com verificações numéricas independentes para garantir a consistência da implementação.

In [ ]:
# Manipulação e análise dos dados
import numpy as np
import pandas as pd

# Exibição das tabelas no notebook
from IPython.display import display

# Configurações de visualização
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 6)

In [ ]:
# URL direta do CSV no GitHub
URL = "https://raw.githubusercontent.com/JoaoIvo15/projeto-pnl-busca-linear/main/data/market_data_ibov_vale3_2020_2025.csv"

# Carregamento dos dados
dados = pd.read_csv(URL, sep=None, engine="python")

# Inspeção inicial
print("Dimensões:", dados.shape)
print("\nInformações gerais:")
dados.info()

print("\nPrimeiras observações:")
display(dados.head())

print("\nÚltimas observações:")
display(dados.tail())

print("\nAmostra aleatória de 10 observações:")
display(dados.sample(n=10, random_state=42))

Dimensões: (1266, 3)

Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1266 entries, 0 to 1265
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Date               1266 non-null   object
 1   IBOV_Close_Points  1266 non-null   object
 2   VALE3_Close        1266 non-null   object
dtypes: object(3)
memory usage: 29.8+ KB

Primeiras observações:


,Date,IBOV_Close_Points,VALE3_Close
0,01/12/2020,"111.399,91","81,25"
1,02/12/2020,"111.878,53","79,84"
2,03/12/2020,"112.291,59","78,96"
3,04/12/2020,"113.750,22","81,98"
4,07/12/2020,"113.589,77","82,95"



Últimas observações:


,Date,IBOV_Close_Points,VALE3_Close
1261,22/12/2025,"158.141,65","72,92"
1262,23/12/2025,"160.455,83","72,90"
1263,26/12/2025,"160.896,64","73,12"
1264,29/12/2025,"160.490,30","72,12"
1265,30/12/2025,"161.125,37","71,96"



Amostra aleatória de 10 observações:


,Date,IBOV_Close_Points,VALE3_Close
1210,09/10/2025,"141.708,19","59,11"
1039,04/02/2025,"125.147,42","54,02"
422,15/08/2022,"113.031,98","68,30"
1118,02/06/2025,"136.786,65","52,56"
848,02/05/2024,"127.122,25","63,92"
781,24/01/2024,"127.815,70","69,90"
23,07/01/2021,"122.385,92","102,32"
405,21/07/2022,"99.033,17","68,57"
43,05/02/2021,"120.240,26","92,69"
581,04/04/2023,"101.869,45","78,04"


## 2. Preparação dos dados

Nesta etapa, os dados serão preparados para os cálculos
matemáticos posteriores.

A base foi obtida a partir de séries históricas do
Ibovespa e da VALE3. Como os valores numéricos foram
exportados no formato brasileiro, será necessário
convertê-los para tipos numéricos reconhecidos pelo
Python. A coluna de datas também será convertida para
o tipo apropriado.

A preparação será realizada em uma cópia da base original,
preservando os dados carregados inicialmente.

Após as conversões, será verificada a consistência dos
tipos de dados e da ordenação cronológica, permitindo
prosseguir para o cálculo dos retornos.

In [ ]:
# Preservar os dados originais
dados_brutos = dados.copy()

# Criar a base que será utilizada na modelagem
dados_modelagem = dados_brutos.copy()


# Converter números no formato brasileiro para float
def converter_numero_brasileiro(serie):
    texto = serie.astype("string").str.strip()

    # Remover símbolo monetário e espaços, se presentes
    texto = (
        texto
        .str.replace("R$", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    # Converter separadores brasileiros:
    # "111.399,91" -> "111399.91"
    formato_brasileiro = texto.str.contains(",", na=False)

    texto.loc[formato_brasileiro] = (
        texto.loc[formato_brasileiro]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(texto, errors="raise")


# Converter a coluna de datas
dados_modelagem["Date"] = pd.to_datetime(
    dados_modelagem["Date"],
    format="%d/%m/%Y",
    errors="raise"
)

# Converter as colunas de preços
colunas_numericas = [
    "IBOV_Close_Points",
    "VALE3_Close"
]

for coluna in colunas_numericas:
    dados_modelagem[coluna] = converter_numero_brasileiro(
        dados_modelagem[coluna]
    )


# Verificações após a conversão
assert dados_modelagem["Date"].is_monotonic_increasing, (
    "As datas não estão em ordem cronológica."
)

assert all(
    pd.api.types.is_numeric_dtype(dados_modelagem[coluna])
    for coluna in colunas_numericas
), "Uma ou mais colunas de preços não foram convertidas corretamente."

print("Preparação concluída.")
print("\nTipos de dados após a conversão:")
print(dados_modelagem.dtypes)

print("\nAmostra dos dados preparados:")
display(dados_modelagem.head())

Preparação concluída.

Tipos de dados após a conversão:
Date                 datetime64[ns]
IBOV_Close_Points           Float64
VALE3_Close                 Float64
dtype: object

Amostra dos dados preparados:


,Date,IBOV_Close_Points,VALE3_Close
0,2020-12-01,111399.91,81.25
1,2020-12-02,111878.53,79.84
2,2020-12-03,112291.59,78.96
3,2020-12-04,113750.22,81.98
4,2020-12-07,113589.77,82.95


## 3. Construção das séries de retornos

A modelagem utilizará os retornos diários da VALE3 e do
Ibovespa, calculados a partir dos respectivos preços de
fechamento.

Para cada série, será utilizado o retorno percentual simples:

$$
R_t = 100\left(\frac{P_t}{P_{t-1}}-1\right),
$$

em que $P_t$ representa o preço de fechamento na data $t$.

O período de análise compreende janeiro de 2021 a dezembro
de 2025. Os dados de dezembro de 2020 serão preservados
como referência para o cálculo do primeiro retorno da
janela selecionada.

Após o cálculo, as duas séries serão organizadas por data,
formando os pares de observações utilizados na modelagem.

In [ ]:
# Calcular os retornos antes de selecionar o período
base_retornos = (
    dados_modelagem
    .sort_values("Date")
    .copy()
)

base_retornos["IBOV_Return_pct"] = (
    base_retornos["IBOV_Close_Points"]
    .pct_change(fill_method=None)
    * 100
)

base_retornos["VALE3_Return_pct"] = (
    base_retornos["VALE3_Close"]
    .pct_change(fill_method=None)
    * 100
)

# Selecionar o período da modelagem
inicio = pd.Timestamp("2021-01-01")
fim = pd.Timestamp("2025-12-31")

mascara_periodo = base_retornos["Date"].between(inicio, fim)

dados_retornos = (
    base_retornos.loc[
        mascara_periodo,
        ["Date", "IBOV_Return_pct", "VALE3_Return_pct"]
    ]
    .copy()
    .reset_index(drop=True)
)

# Não descartar silenciosamente observações incompletas
colunas_retornos = [
    "IBOV_Return_pct",
    "VALE3_Return_pct"
]

# Garantir que os retornos sejam numéricos
for coluna in colunas_retornos:
    dados_retornos[coluna] = pd.to_numeric(
        dados_retornos[coluna],
        errors="coerce"
    )

# Verificar valores ausentes após a conversão
ausencias = dados_retornos[colunas_retornos].isna().sum()

if ausencias.any():
    raise ValueError(
        "Existem retornos ausentes ou não numéricos:\n"
        f"{ausencias}"
    )

# Converter explicitamente para um array NumPy de float
retornos_array = dados_retornos[
    colunas_retornos
].to_numpy(dtype=np.float64)

# Verificar se todos os retornos são finitos
if not np.isfinite(retornos_array).all():
    raise ValueError(
        "Foram encontrados retornos não finitos."
    )

# Verificar duplicidade de datas
if dados_retornos["Date"].duplicated().any():
    raise ValueError(
        "Existem datas duplicadas na amostra."
    )

print(
    "Período:",
    dados_retornos["Date"].min().date(),
    "a",
    dados_retornos["Date"].max().date()
)

print("Número de observações:", len(dados_retornos))

display(dados_retornos.head())
display(
    dados_retornos[
        ["IBOV_Return_pct", "VALE3_Return_pct"]
    ].describe().T
)

Período: 2021-01-04 a 2025-12-30
Número de observações: 1246


,Date,IBOV_Return_pct,VALE3_Return_pct
0,2021-01-04,-0.13656,4.585477
1,2021-01-05,0.438771,1.683796
2,2021-01-06,-0.231311,2.806452
3,2021-01-07,2.75889,7.018094
4,2021-01-08,2.198545,-0.312744


,count,mean,std,min,25%,50%,75%,max
IBOV_Return_pct,1246.0,0.030561,1.117516,-4.866,-0.616343,0.042055,0.720555,5.541483
VALE3_Return_pct,1246.0,0.00303,1.935462,-9.413385,-1.098583,-0.04904,0.990055,10.395708


## 4. Formulação matemática do problema

O objetivo desta etapa é estudar a relação histórica entre
os retornos diários da VALE3 e do Ibovespa por meio de um
modelo linear.

Sejam $m_t$ o retorno percentual do Ibovespa e $r_t$
o retorno percentual observado da VALE3 na data $t$.

Adota-se o modelo:

$$
\widehat r_t = \alpha + \beta m_t.
$$

As variáveis de decisão são o intercepto $\alpha$ e o
coeficiente angular $\beta$, reunidos no vetor
$x=(\alpha,\beta)^T\in\mathbb{R}^2$.

Para cada observação, define-se o resíduo como a diferença
entre o retorno estimado e o retorno observado:

$$
e_t(\alpha,\beta)=\alpha+\beta m_t-r_t.
$$

A função objetivo será a soma dos quadrados dos resíduos,
dividida por $2n$, em que $n$ é o número de observações:

$$
f(\alpha,\beta)=
\frac{1}{2n}\sum_{t=1}^{n}
(\alpha+\beta m_t-r_t)^2.
$$

Assim, o problema de otimização é:

$$
\min_{(\alpha,\beta)\in\mathbb{R}^2}
f(\alpha,\beta).
$$

O domínio é $\mathbb{R}^2$, pois não serão impostas
restrições aos valores dos dois parâmetros. As observações
históricas permanecem fixas durante a otimização.

A escolha dessa função permite formular o problema como
uma minimização irrestrita e aplicar os métodos de busca
linear estudados na disciplina.

In [ ]:
# Extrair os vetores de observações da base preparada
m = dados_retornos["IBOV_Return_pct"].to_numpy(dtype=np.float64)
r = dados_retornos["VALE3_Return_pct"].to_numpy(dtype=np.float64)

n = len(m)

# Verificar a consistência dos vetores
if n == 0 or len(r) != n:
    raise ValueError("Os vetores de retornos são incompatíveis.")

if not np.isfinite(m).all() or not np.isfinite(r).all():
    raise ValueError("Existem valores não finitos nos retornos.")


def modelo_linear(x, m_observado):
    """
    Calcula os retornos estimados da VALE3.

    Parâmetros
    ----------
    x : array-like
        Vetor de parâmetros [alpha, beta].
    m_observado : array-like
        Retornos observados do Ibovespa.

    Retorno
    -------
    numpy.ndarray
        Retornos estimados pelo modelo.
    """
    alpha, beta = np.asarray(x, dtype=np.float64)

    return alpha + beta * np.asarray(
        m_observado, dtype=np.float64
    )


print("Número de observações:", n)
print("Número de parâmetros:", 2)
print("Domínio dos parâmetros: R²")

Número de observações: 1246
Número de parâmetros: 2
Domínio dos parâmetros: R²


In [ ]:
def funcao_objetivo(x):
    """
    Calcula a função objetivo de mínimos quadrados.

    f(alpha, beta) = (1 / (2n)) * sum(residuos**2)

    Parâmetros
    ----------
    x : array-like
        Vetor de parâmetros [alpha, beta].

    Retorno
    -------
    float
        Valor escalar da função objetivo.
    """
    # Obter as previsões para os parâmetros candidatos
    retornos_estimados = modelo_linear(x, m)

    # Resíduos: retorno estimado menos retorno observado
    residuos = retornos_estimados - r

    # Soma dos quadrados dos resíduos, normalizada por 2n
    custo = np.sum(residuos**2) / (2 * n)

    return float(custo)

A função objetivo foi implementada a partir do modelo linear
parametrizado e das observações históricas dos retornos.

Para cada par candidato de parâmetros $(\alpha,\beta)$,
o modelo calcula os retornos estimados da VALE3. Em seguida,
são calculados os resíduos em relação aos retornos observados,
e seus quadrados são somados e divididos por $2n$.

Dessa forma, a função objetivo associa um custo escalar a
cada ponto do domínio $\mathbb{R}^2$. O problema consiste
em encontrar parâmetros que minimizem esse custo, utilizando
as direções de busca e os métodos de busca linear estudados
na disciplina.

### Dedução analítica do gradiente e da Hessiana

A função objetivo definida anteriormente é

$$
f(\alpha,\beta)=
\frac{1}{2n}\sum_{t=1}^{n}
(\alpha+\beta m_t-r_t)^2.
$$

Para simplificar a notação, define-se o resíduo da observação
$t$ como

$$
e_t=\alpha+\beta m_t-r_t.
$$

Assim, a função objetivo pode ser escrita como

$$
f(\alpha,\beta)=\frac{1}{2n}\sum_{t=1}^{n}e_t^2.
$$

Como as observações $m_t$ e $r_t$ são constantes durante
a otimização, aplicamos a regra da cadeia para calcular
as derivadas em relação aos parâmetros $\alpha$ e $\beta$.

A derivada parcial em relação a $\alpha$ é

$$
\frac{\partial f}{\partial\alpha}
=
\frac{1}{2n}\sum_{t=1}^{n}
2e_t\frac{\partial e_t}{\partial\alpha}.
$$

Como $\frac{\partial e_t}{\partial\alpha}=1$, obtemos

$$
\frac{\partial f}{\partial\alpha}
=
\frac{1}{n}\sum_{t=1}^{n}e_t.
$$

Analogamente, como
$\frac{\partial e_t}{\partial\beta}=m_t$, temos

$$
\frac{\partial f}{\partial\beta}
=
\frac{1}{2n}\sum_{t=1}^{n}
2e_t m_t
=
\frac{1}{n}\sum_{t=1}^{n}m_t e_t.
$$

Portanto, o gradiente da função objetivo é

$$
\nabla f(\alpha,\beta)=
\begin{bmatrix}
\frac{1}{n}\sum_{t=1}^{n}e_t\\[6pt]
\frac{1}{n}\sum_{t=1}^{n}m_t e_t
\end{bmatrix}.
$$

Para obter a Hessiana, derivamos novamente os componentes
do gradiente. Como $\alpha$ e $\beta$ são as variáveis de
decisão, enquanto $m_t$ e $r_t$ permanecem fixos, resulta

$$
\frac{\partial^2 f}{\partial\alpha^2}=1,
\qquad
\frac{\partial^2 f}{\partial\alpha\partial\beta}
=
\frac{1}{n}\sum_{t=1}^{n}m_t.
$$

Da mesma forma,

$$
\frac{\partial^2 f}{\partial\beta\partial\alpha}
=
\frac{1}{n}\sum_{t=1}^{n}m_t,
$$

e

$$
\frac{\partial^2 f}{\partial\beta^2}
=
\frac{1}{n}\sum_{t=1}^{n}m_t^2.
$$

Logo, a matriz Hessiana é

$$
\nabla^2 f(\alpha,\beta)=
\begin{bmatrix}
1 & \frac{1}{n}\sum_{t=1}^{n}m_t\\[6pt]
\frac{1}{n}\sum_{t=1}^{n}m_t &
\frac{1}{n}\sum_{t=1}^{n}m_t^2
\end{bmatrix}.
$$

Observa-se que a Hessiana não depende dos valores de
$\alpha$ e $\beta$. Portanto, ela é constante para a
amostra utilizada, embora seus elementos dependam dos
retornos observados do Ibovespa.

In [ ]:
def gradiente(x):
    """
    Calcula o gradiente analítico da função objetivo.

    Parameters
    ----------
    x : array-like, shape (2,)
        Vetor de parâmetros [alpha, beta].

    Returns
    -------
    numpy.ndarray, shape (2,)
        Vetor gradiente da função objetivo.
    """
    x = np.asarray(x, dtype=np.float64)

    if x.shape != (2,):
        raise ValueError(
            "O vetor de parâmetros deve ter dimensão 2."
        )

    alpha, beta = x

    # Resíduos para o par de parâmetros avaliado
    residuos = alpha + beta * m - r

    # Derivadas parciais em relação a alpha e beta
    return np.array([
        np.mean(residuos),
        np.mean(m * residuos)
    ])


def hessiana():
    """
    Calcula a matriz Hessiana analítica da função objetivo.

    Returns
    -------
    numpy.ndarray, shape (2, 2)
        Matriz Hessiana constante para a amostra.
    """
    return np.array([
        [1.0, np.mean(m)],
        [np.mean(m), np.mean(m**2)]
    ])


print("Dimensão do gradiente: 2")
print("Gradiente no ponto de avaliação (0, 1):")

display(gradiente(np.array([0.0, 1.0])))

print("\nMatriz Hessiana:")

display(pd.DataFrame(
    hessiana(),
    index=["alpha", "beta"],
    columns=["alpha", "beta"]
))

Dimensão do gradiente: 2
Gradiente no ponto de avaliação (0, 1):


array([0.02753063, 0.2201058 ])


Matriz Hessiana:


,alpha,beta
alpha,1.000000,0.030561
beta,0.030561,1.248773


### Análise de convexidade

A convexidade da função objetivo será analisada por meio
da matriz Hessiana obtida anteriormente.

A Hessiana é simétrica e constante para a amostra utilizada.
Como se trata de uma matriz de ordem dois, podemos verificar
sua positividade definida pelo critério de Sylvester.

Para isso, é necessário que seus menores principais líderes
sejam positivos. Neste caso, a primeira condição é

$$
H_{11}=1>0.
$$

A segunda condição é

$$
\det(H)=
\frac{1}{n}\sum_{t=1}^{n}m_t^2
-
\left(\frac{1}{n}\sum_{t=1}^{n}m_t\right)^2
>0.
$$

O determinante corresponde à variância dos retornos do
Ibovespa na amostra, calculada com divisor $n$.

Se essa variância for estritamente positiva, a Hessiana
será positiva definida. Consequentemente, a função objetivo
será estritamente convexa em $\mathbb{R}^2$ e possuirá um
único minimizador global.

In [ ]:
# Obter a Hessiana calculada analiticamente
H = hessiana()

# Menores principais e propriedades da matriz
menor_principal_1 = H[0, 0]
det_H = np.linalg.det(H)
autovalores = np.linalg.eigvalsh(H)

# A variância utiliza divisor n, conforme a expressão analítica
variancia_ibov = np.var(m, ddof=0)

print("Hessiana:")
display(pd.DataFrame(
    H,
    index=["alpha", "beta"],
    columns=["alpha", "beta"]
))

print(f"Primeiro menor principal: {menor_principal_1:.10f}")
print(f"Determinante da Hessiana: {det_H:.10f}")
print(f"Variância dos retornos do Ibovespa: {variancia_ibov:.10f}")
print("Autovalores da Hessiana:", autovalores)

# Verificações matemáticas
assert np.allclose(H, H.T), "A Hessiana não é simétrica."

assert np.isclose(
    det_H, variancia_ibov, rtol=1e-10, atol=1e-12
), "O determinante não corresponde à variância calculada."

if menor_principal_1 > 0 and det_H > 0:
    print(
        "\nConclusão: Hessiana positiva definida. "
        "A função objetivo é estritamente convexa "
        "e possui um único minimizador global."
    )
else:
    print(
        "\nA Hessiana não é positiva definida. "
        "É necessário investigar as propriedades da amostra."
    )

Hessiana:


,alpha,beta
alpha,1.000000,0.030561
beta,0.030561,1.248773


Primeiro menor principal: 1.0000000000
Determinante da Hessiana: 1.2478388161
Variância dos retornos do Ibovespa: 1.2478388161
Autovalores da Hessiana: [0.99630074 1.25247203]

Conclusão: Hessiana positiva definida. A função objetivo é estritamente convexa e possui um único minimizador global.


### Escolha do ponto inicial e direção de descida

Para executar o primeiro passo de busca linear, será
adotado o ponto inicial

$$
x_0=
\begin{bmatrix}
0\\
1
\end{bmatrix}
$$

Essa escolha representa uma referência inicial simples:
intercepto nulo e coeficiente angular unitário.

A partir desse ponto, calcula-se o gradiente da função
objetivo e define-se a direção de descida mais íngreme:

$$
d_0=-\nabla f(x_0).
$$

Para confirmar que a direção é efetivamente de descida,
verifica-se a condição

$$
\nabla f(x_0)^T d_0<0.
$$

Em seguida, será aplicada a Regra de Armijo para determinar
um comprimento de passo positivo que produza redução
suficiente da função objetivo.

In [ ]:
# Ponto inicial: intercepto zero e inclinação unitária
x0 = np.array([0.0, 1.0], dtype=np.float64)

# Avaliar a função objetivo e o gradiente no ponto inicial
f0 = funcao_objetivo(x0)
g0 = gradiente(x0)

# Direção do gradiente negativo
d0 = -g0

# Derivada direcional da função objetivo em d0
derivada_direcional = float(np.dot(g0, d0))

print("Ponto inicial x0:", x0)
print("Valor inicial da função:", f0)
print("Gradiente em x0:", g0)
print("Direção de descida d0:", d0)
print("Gradiente transposto vezes direção:", derivada_direcional)

# Verificação da condição de descida estrita
assert derivada_direcional < 0, (
    "A direção calculada não satisfaz a condição estrita de descida."
)

print("\nDireção de descida confirmada.")

Ponto inicial x0: [0. 1.]
Valor inicial da função: 1.4672271589800239
Gradiente em x0: [0.02753063 0.2201058 ]
Direção de descida d0: [-0.02753063 -0.2201058 ]
Gradiente transposto vezes direção: -0.049204499730869475

Direção de descida confirmada.


### Determinação do comprimento do passo pela Regra de Armijo

A Regra de Armijo é um método de busca linear inexata
que procura um comprimento de passo suficiente para
reduzir a função objetivo.

Partindo do comprimento inicial $\alpha_0=1$, o método
avalia sucessivamente o ponto

$$
x_0+\alpha d_0.
$$

O passo é aceito quando satisfaz a condição de Armijo:

$$
f(x_0+\alpha d_0)
\leq
f(x_0)+c\alpha\nabla f(x_0)^T d_0,
$$

em que $c\in(0,1)$ controla a redução suficiente exigida.
Quando a condição não é satisfeita, o comprimento do
passo é multiplicado por um fator $\rho\in(0,1)$,
e a avaliação é repetida.

Neste experimento, serão utilizados $c=10^{-4}$ e
$\rho=0{,}5$. Esses parâmetros serão mantidos explícitos
na implementação para permitir a reprodução dos resultados.

In [ ]:
def busca_armijo(
    funcao,
    x,
    direcao,
    gradiente_x,
    alpha_inicial=1.0,
    rho=0.5,
    c=1e-4,
    max_iter=100,
    alpha_minimo=1e-14,
):
    """
    Determina o comprimento do passo pela Regra de Armijo.

    A condição de aceitação é:
        f(x + alpha*d) <= f(x) + c*alpha*gradiente_x.T*d

    Retorna o passo aceito, o novo ponto, o novo custo
    e o número de tentativas realizadas.
    """

    x = np.asarray(x, dtype=np.float64)
    direcao = np.asarray(direcao, dtype=np.float64)
    gradiente_x = np.asarray(gradiente_x, dtype=np.float64)

    if x.shape != direcao.shape or x.shape != gradiente_x.shape:
        raise ValueError(
            "O ponto, a direção e o gradiente devem ter a mesma dimensão."
        )

    if alpha_inicial <= 0 or alpha_minimo <= 0:
        raise ValueError("Os comprimentos de passo devem ser positivos.")

    if not 0 < rho < 1:
        raise ValueError("rho deve pertencer ao intervalo (0, 1).")

    if not 0 < c < 1:
        raise ValueError("c deve pertencer ao intervalo (0, 1).")

    # Derivada direcional na direção fornecida
    inclinacao = float(np.dot(gradiente_x, direcao))

    if inclinacao >= 0:
        raise ValueError(
            "A direção informada não é de descida estrita."
        )

    # Valor da função no ponto inicial
    f_inicial = float(funcao(x))
    alpha = float(alpha_inicial)

    # Busca por redução suficiente
    for tentativa in range(1, max_iter + 1):

        x_candidato = x + alpha * direcao
        f_candidato = float(funcao(x_candidato))

        lado_direito = f_inicial + c * alpha * inclinacao

        # Aceitar o passo se satisfizer a condição de Armijo
        if (
            np.isfinite(f_candidato)
            and f_candidato <= lado_direito
        ):
            return {
                "alpha": alpha,
                "x_novo": x_candidato,
                "f_inicial": f_inicial,
                "f_novo": f_candidato,
                "tentativas": tentativa,
                "derivada_direcional": inclinacao,
                "lado_direito_armijo": lado_direito,
            }

        # Reduzir o comprimento do passo
        alpha *= rho

        if alpha < alpha_minimo:
            break

    raise RuntimeError(
        "A busca de Armijo não encontrou um passo aceitável."
    )

In [ ]:
# Parâmetros utilizados na busca
parametros_armijo = {
    "alpha_inicial": 1.0,
    "rho": 0.5,
    "c": 1e-4,
    "max_iter": 100,
}

# Executar a busca linear
resultado_armijo = busca_armijo(
    funcao=funcao_objetivo,
    x=x0,
    direcao=d0,
    gradiente_x=g0,
    **parametros_armijo,
)

alpha_aceito = resultado_armijo["alpha"]
x1 = resultado_armijo["x_novo"]
f1 = resultado_armijo["f_novo"]

# Verificar redução efetiva do custo
assert f1 < f0, (
    "O custo não diminuiu: a redução exigida não foi comprovada."
)

# Verificar novamente a condição de Armijo
assert f1 <= resultado_armijo["lado_direito_armijo"]

print("PONTO INICIAL")
print("x0 =", x0)
print(f"f(x0) = {f0:.12f}")

print("\nDIREÇÃO DE DESCIDA")
print("d0 =", d0)
print(
    "Gradiente transposto vezes d0 =",
    resultado_armijo["derivada_direcional"],
)

print("\nRESULTADO DA BUSCA DE ARMIJO")
print(f"Alpha aceito = {alpha_aceito:.12f}")
print("Tentativas =", resultado_armijo["tentativas"])
print("x1 =", x1)
print(f"f(x1) = {f1:.12f}")

print("\nREDUÇÃO DO CUSTO")
print(f"f(x0) - f(x1) = {f0 - f1:.12f}")
print("Redução comprovada:", f1 < f0)

PONTO INICIAL
x0 = [0. 1.]
f(x0) = 1.467227158980

DIREÇÃO DE DESCIDA
d0 = [-0.02753063 -0.2201058 ]
Gradiente transposto vezes d0 = -0.049204499730869475

RESULTADO DA BUSCA DE ARMIJO
Alpha aceito = 1.000000000000
Tentativas = 1
x1 = [-0.02753063  0.7798942 ]
f(x1) = 1.448836189716

REDUÇÃO DO CUSTO
f(x0) - f(x1) = 0.018390969265
Redução comprovada: True


A partir do ponto inicial escolhido, foi calculada a direção
do gradiente negativo e verificada a condição de descida.
A Regra de Armijo determinou um comprimento de passo
aceitável, produzindo o novo ponto $x_1=x_0+\alpha_0d_0$.

A redução observada entre $f(x_0)$ e $f(x_1)$ confirma
numericamente que o primeiro passo diminuiu a função
objetivo, conforme exigido no enunciado do projeto.